##  Word Analogy Function

### Intuition
When Word2Vec is trained, each word is mapped to a point in high-dimensional space.
The geometry of that space captures **meaning** — specifically, relationships between
words appear as consistent *directions* (vectors).

For example:
- The direction **man → king** is roughly the same as **woman → queen**
- The direction **Paris → France** is roughly the same as **Berlin → Germany**

This means the *difference* between two word vectors encodes their relationship.

---

### The Math
Given three words **A, B, A\***, we want to find **B\*** such that:

> *"A is to B as A\* is to B\*"*

We compute:

$$\vec{B^*} \approx \vec{B} - \vec{A} + \vec{A^*}$$

Then we search the vocabulary for the word whose vector is **closest** to this target
(using cosine similarity), excluding the three input words from the results.

---

### Concrete Example

$$\vec{\text{king}} - \vec{\text{man}} + \vec{\text{woman}} \approx \vec{\text{queen}}$$

Step by step:
1. $\vec{\text{king}} - \vec{\text{man}}$ → extracts the **"royalty"** direction, stripping away the male component
2. $+ \vec{\text{woman}}$ → adds the **female** component back
3. The result lands near $\vec{\text{queen}}$ in the embedding space

This is never explicitly learned — the geometry **emerges naturally** from co-occurrence
patterns in the training text.

In [1]:
from load import load_embeddings_and_vocab

embeddings, word2idx, idx2word = load_embeddings_and_vocab()

In [2]:
import numpy as np

def word_analogy(a, b, a_star, embeddings=embeddings, word2idx=word2idx, idx2word=idx2word, top_k=5):
    """
    Solves: a is to b as a_star is to ?
    Formula: target = vec(b) - vec(a) + vec(a_star)
    """
    a      = a.lower()
    b      = b.lower()
    a_star = a_star.lower()

    for word in [a, b, a_star]:
        if word not in word2idx:
            raise ValueError(f"'{word}' not in vocabulary")

    # Get vectors
    vec_a      = embeddings[word2idx[a]]
    vec_b      = embeddings[word2idx[b]]
    vec_a_star = embeddings[word2idx[a_star]]

    # Compute analogy target vector
    target_vec = vec_b - vec_a + vec_a_star

    # Normalize using numpy
    norms      = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-10
    emb_norm   = embeddings / norms
    target_norm = target_vec / (np.linalg.norm(target_vec) + 1e-10)

    # Cosine similarity with every word
    similarities = emb_norm @ target_norm   # (VOCAB_SIZE,)

    # Exclude input words
    for word in [a, b, a_star]:
        similarities[word2idx[word]] = -np.inf

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = [(idx2word[i], similarities[i]) for i in top_indices]
    return results

In [ ]:

# Analogy: Rome : Italy :: Madrid : ?
result = word_analogy("Rome", "Italy", "Madrid")

print("Analogy: 'Rome' is to 'Italy' as 'Madrid' is to ?")
for i, (word, score) in enumerate(result, 1):
    print(f"{i}. {word} -> {score:.4f}")
print("=" * 50)

result = word_analogy("",  "doctor", "woman")  # → state / official
print("Analogy: 'Man' is to 'Doctor' as 'Woman' is to ?")
for i, (word, score) in enumerate(result, 1):
    print(f"{i}. {word} -> {score:.4f}")
print("=" * 50)



Analogy: 'Rome' is to 'Italy' as 'Madrid' is to ?
1. bibiana -> 0.6544
2. isolde -> 0.6179
3. perez -> 0.6172
4. secretario -> 0.6169
5. ottoz -> 0.6122


ValueError: 'man ' not in vocabulary